# Telco Customer Churn — Phase 1: Data Cleaning

**Dataset:** IBM Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn.csv`)  
**Objective:** Load the raw dataset, profile its structure and quality, resolve any issues, and export a clean file ready for EDA.

| Section | Description |
|---------|-------------|
| 0 | Setup & imports |
| 1 | Data loading |
| 2 | Initial profiling |
| 3 | Data quality assessment |
| 4 | Issue investigation — `TotalCharges` |
| 5 | Cleaning decisions |
| 6 | Data transformation |
| 7 | Post-cleaning validation |
| 8 | Final QA report |
| 9 | Before vs After |
| 10 | Export & verify |
| — | Phase 1 summary |

## 0 · Setup

### Analytical Grain

**One row = one customer**

The dataset is analyzed at the customer level. Each `customerID` represents one customer record. This grain is maintained throughout the cleaning process.

In [ ]:
import pandas as pd

## 1 · Data Loading

Load the raw CSV from the project's `data/raw/` directory and take a first look at its shape and columns.

> **Repository layout assumed:**
> ```
> telco-churn-analysis/
> ├── data/
> │   ├── raw/
> │   │   └── WA_Fn-UseC_-Telco-Customer-Churn.csv
> │   └── processed/
> └── notebooks/
>     └── 01_data_cleaning.ipynb   ← run from here
> ```

In [ ]:
DATA_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head()

In [ ]:
# Full column list
df.columns.tolist()

## 2 · Initial Profiling

Examine dtypes, null counts, and basic descriptive statistics before any cleaning.

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Null values per column
df.isnull().sum()

In [ ]:
# Whitespace-only cells (can masquerade as non-null)
(df == " ").sum()

In [ ]:
# Descriptive statistics for numeric columns
df.describe()

## 3 · Data Quality Assessment

Systematic checks across all quality dimensions before any cleaning is performed.

In [ ]:
# Duplicate rows and customer IDs
print("Duplicate rows:       ", df.duplicated().sum())
print("Duplicate customerIDs:", df['customerID'].duplicated().sum())

In [ ]:
# Categorical value audit
cat_cols = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod", "Churn",
]

for col in cat_cols:
    print(f"\n{'─'*40}")
    print(f"  {col}")
    print(f"{'─'*40}")
    print(df[col].value_counts().to_string())

In [ ]:
# Numeric range audit — explicit constraint tests
numeric_validation = {
    "Invalid SeniorCitizen": int(
        (~df["SeniorCitizen"].isin([0, 1])).sum()
    ),
    "Invalid tenure (< 0 or > 72)": int(
        ((df["tenure"] < 0) | (df["tenure"] > 72)).sum()
    ),
    "Invalid MonthlyCharges (< 0)": int(
        (df["MonthlyCharges"] < 0).sum()
    ),
    "Invalid TotalCharges (< 0)": int(
        pd.to_numeric(df["TotalCharges"], errors="coerce").lt(0).sum()
    ),
}

pd.Series(numeric_validation)

In [ ]:
# Cross-field logical consistency check
# Customers with no internet service should have 'No internet service'
# for all internet-dependent columns.
internet_dependent_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]

no_internet_mask = df["InternetService"] == "No"
logical_inconsistencies = 0

for col in internet_dependent_cols:
    invalid = df[no_internet_mask & (df[col] != "No internet service")]
    logical_inconsistencies += len(invalid)
    status = f"{len(invalid)} inconsistent records" if len(invalid) else "✓ clean"
    print(f"{col:<20}  {status}")

print(f"\nTotal logical inconsistencies: {logical_inconsistencies}")

## 4 · Issue Investigation — `TotalCharges`

`df.info()` shows `TotalCharges` stored as `object` despite representing a monetary measure. We first quantify blank values, then inspect the affected records.

In [ ]:
# Preview the suspicious column
df[["tenure", "MonthlyCharges", "TotalCharges"]].head(10)

In [ ]:
# Count whitespace-only values
print("Blank TotalCharges:  ", df["TotalCharges"].astype(str).str.strip().eq("").sum())
print("Blank MonthlyCharges:", df["MonthlyCharges"].astype(str).str.strip().eq("").sum())

In [ ]:
# Inspect the 11 affected records
blank_mask = df["TotalCharges"].astype(str).str.strip().eq("")
df.loc[blank_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

### Interpretation

The 11 whitespace-only `TotalCharges` records were investigated individually.

All 11 affected records have `tenure = 0`. Since `TotalCharges` represents cumulative customer charges and these records have zero recorded tenure, the whitespace-only values are treated as **0** for this analysis.

This is an analytical assumption rather than a claim about the underlying billing system.

## 5 · Cleaning Decisions

| Issue | Evidence | Action |
|---|---|---|
| `TotalCharges` stored as `object` | Column represents a numeric monetary measure | Convert to `float64` |
| 11 whitespace-only `TotalCharges` values | All affected records have `tenure = 0` | Treat as 0 for this analysis |
| Duplicate rows | 0 identified | No action |
| Duplicate customer IDs | 0 identified | No action |
| Unexpected categorical values | None identified | No action |
| Invalid numeric values | None identified | No action |
| Logical inconsistencies | None identified | No action |

## 6 · Data Transformation

Apply the single correction identified above: replace whitespace-only `TotalCharges` entries with `0` and cast the column to `float64`.

In [ ]:
# Replace whitespace-only strings with '0', then cast to numeric
df["TotalCharges"] = df["TotalCharges"].replace(r"^\s*$", "0", regex=True)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])

print("New dtype:", df["TotalCharges"].dtype)
print("Remaining NaN:", df["TotalCharges"].isna().sum())

## 7 · Post-Cleaning Validation

Confirm the fix is correct and that no unintended side-effects occurred.

In [ ]:
# Verify tenure=0 records now hold 0.0 for TotalCharges
df.loc[
    df["tenure"] == 0,
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]

In [ ]:
# Confirm no remaining blank or null values
print("Blank TotalCharges:", df["TotalCharges"].astype(str).str.strip().eq("").sum())
print("Null  TotalCharges:", df["TotalCharges"].isna().sum())
print("Row count unchanged:", df.shape[0])

## 8 · Final QA Report

Consolidated pass/fail summary across all quality dimensions, including the logical consistency check from Section 3.

In [ ]:
qa_results = {
    "Rows":                        df.shape[0],
    "Columns":                     df.shape[1],
    "Null values":                 int(df.isna().sum().sum()),
    "Duplicate rows":              int(df.duplicated().sum()),
    "Duplicate customer IDs":      int(df["customerID"].duplicated().sum()),
    "TotalCharges dtype":          str(df["TotalCharges"].dtype),
    "Blank TotalCharges":          int(df["TotalCharges"].astype(str).str.strip().eq("").sum()),
    "Logical inconsistencies":     logical_inconsistencies,
    "Invalid SeniorCitizen":       numeric_validation["Invalid SeniorCitizen"],
    "Invalid tenure":              numeric_validation["Invalid tenure (< 0 or > 72)"],
    "Invalid MonthlyCharges":      numeric_validation["Invalid MonthlyCharges (< 0)"],
    "Invalid TotalCharges":        numeric_validation["Invalid TotalCharges (< 0)"],
}

pd.Series(qa_results)

## 9 · Before vs After

| Metric | Before Cleaning | After Cleaning |
|---|---:|---:|
| Rows | 7,043 | 7,043 |
| Columns | 21 | 21 |
| `TotalCharges` dtype | `object` | `float64` |
| Blank `TotalCharges` | 11 | 0 |
| Null values | 0 | 0 |
| Duplicate rows | 0 | 0 |
| Duplicate customer IDs | 0 | 0 |
| Logical inconsistencies | 0 | 0 |

## 10 · Export & Verify

Write the cleaned dataset to disk, then reload and confirm its integrity.

In [ ]:
OUTPUT_PATH = "../data/processed/telco_cleaned.csv"

df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved → {OUTPUT_PATH}")

In [ ]:
# Reload and verify
cleaned_df = pd.read_csv(OUTPUT_PATH)

assert cleaned_df.shape == (7043, 21), "Unexpected shape after export"
assert cleaned_df["TotalCharges"].dtype == float, "TotalCharges should be float"
assert cleaned_df.isna().sum().sum() == 0, "Unexpected nulls after export"
assert cleaned_df.duplicated().sum() == 0, "Unexpected duplicate rows after export"
assert cleaned_df["customerID"].duplicated().sum() == 0, "Unexpected duplicate customer IDs after export"
assert cleaned_df["TotalCharges"].astype(str).str.strip().eq("").sum() == 0, "Blank TotalCharges remain after export"

print("All assertions passed.")
print(f"Shape:                    {cleaned_df.shape}")
print(f"TotalCharges dtype:       {cleaned_df['TotalCharges'].dtype}")
print(f"Null values:              {cleaned_df.isna().sum().sum()}")
print(f"Duplicate rows:           {cleaned_df.duplicated().sum()}")
print(f"Duplicate customer IDs:   {cleaned_df['customerID'].duplicated().sum()}")
print(f"Blank TotalCharges:       {cleaned_df['TotalCharges'].astype(str).str.strip().eq('').sum()}")

---

# Phase 1 — Data Cleaning Complete

## Summary

The Telco Customer Churn dataset contains **7,043 customer records and 21 columns** at customer-level grain.

The data-quality assessment covered:

- Missing values
- Blank/whitespace values
- Data types
- Categorical values
- Numeric validity
- Duplicate records
- Duplicate customer IDs
- Cross-field logical consistency

### Primary Issue Identified

`TotalCharges` was stored as `object` and contained **11 whitespace-only values**.

All 11 affected records had `tenure = 0`. These values were treated as 0 for this analysis and the column was converted to `float64`.

### Final Validation

All planned quality checks passed with no additional data corrections required.

The cleaned dataset was exported to:

`data/processed/telco_cleaned.csv`

**Phase 1 — Data Cleaning: COMPLETE ✅**